# 6 - Monthly raw-event distribution by platform

Đọc dữ liệu raw (đã partition sang parquet trong `data/lake/raw_events`) cho từng tháng và so sánh
phân phối giữa **android** và **ios**.

Lưu ý về nguồn dữ liệu (đã kiểm tra qua `partition_manifest.json`):

| Tháng | Nguồn parquet | Platform có trong file |
|---|---|---|
| T3 | `march/` (`android_events_t3-2026.csv`, `ios_events_t3-2026.csv`) | android + ios |
| T6 | `678/*/` (`log_events_t6.csv`) | chỉ android |
| T7 | `july/` (`android_events_all-t7.csv`, `ios_events_all-t7.csv`) | android + ios |
| T8 | `678/*/` (`log_events_t8.csv`) | chỉ android |

`log_events_t7.csv` trong `678/` chỉ có ios và trùng gần như hoàn toàn với `ios_events_all-t7.csv`,
nên mặc định dùng `july/` cho T7 để tránh đếm trùng. Có thể đổi trong `SOURCES` bên dưới.

`client_time` không đồng nhất: T3 là ISO string (UTC), T6-T8 là epoch milliseconds. Notebook parse cả hai
và gắn cờ các event có timestamp lỗi / nằm ngoài tháng.

Toàn bộ aggregation chạy bằng DuckDB trực tiếp trên parquet (~230M rows), không load raw vào pandas.

## Cấu hình

In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

WORKING_DIR = Path.cwd()
REPO_ROOT = WORKING_DIR if (WORKING_DIR / "data" / "lake").exists() else WORKING_DIR.parent
LAKE = REPO_ROOT / "data" / "lake" / "raw_events"

# month label -> list of (parquet glob, optional _source_file filter)
SOURCES = {
    "2026-03": [(LAKE / "march" / "platform=*" / "bucket=*" / "*.parquet", None)],
    "2026-06": [(LAKE / "678" / "*" / "platform=*" / "bucket=*" / "*.parquet", "log_events_t6.csv")],
    "2026-07": [(LAKE / "july" / "platform=*" / "bucket=*" / "*.parquet", None)],
    "2026-08": [(LAKE / "678" / "*" / "platform=*" / "bucket=*" / "*.parquet", "log_events_t8.csv")],
}
MONTHS = list(SOURCES)
PLATFORMS = ["android", "ios"]
PALETTE = {"android": "#3DDC84", "ios": "#555555"}
LOCAL_TZ_OFFSET_HOURS = 7   # client_time is UTC; hour-of-day plots use Vietnam time
TOP_N_SCREENS = 20

con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")
con.execute(f"SET temp_directory = '{REPO_ROOT / 'output' / 'duckdb_tmp'}'")
con.execute("SET preserve_insertion_order = false")

## Tạo view `ev` thống nhất cho tất cả các tháng

- `platform` lấy từ partition (`_partition_platform`), đã được chuẩn hoá lowercase khi partition.
- `ts`: timestamp UTC đã parse; `NULL` nếu `client_time` không hợp lệ.
- `in_month`: event có `ts` nằm trong đúng tháng của file.
- `dur`: chỉ có ở T6-T8 (T3 không có cột này).
- Epoch có thể có hậu tố `.0` (vd. `1785506041318.0` ở ios T7) nên parse qua DOUBLE.

In [ ]:
def _source_sql(month: str, glob: Path, source_file: str | None) -> str:
    where = f"WHERE _source_file = '{source_file}'" if source_file else ""
    return f"""
        SELECT '{month}' AS month, *
        FROM read_parquet('{glob.as_posix()}', union_by_name = true)
        {where}
    """

union_sql = "\nUNION ALL BY NAME\n".join(
    _source_sql(m, g, f) for m, parts in SOURCES.items() for g, f in parts
)

con.execute(f"""
CREATE OR REPLACE VIEW ev AS
WITH raw AS ({union_sql}),
parsed AS (
    SELECT
        month,
        _partition_platform AS platform,
        session_id,
        device_id,
        NULLIF(regexp_replace(customer_id, '\\.0$', ''), '') AS customer_id,
        key,
        segmentation_name,
        client_time,
        CASE
            WHEN regexp_full_match(client_time, '\\d{{13}}(\\.\\d*)?')
                THEN make_timestamp(CAST(client_time::DOUBLE * 1000 AS BIGINT))
            WHEN regexp_full_match(client_time, '\\d{{10}}(\\.\\d*)?')
                THEN make_timestamp(CAST(client_time::DOUBLE * 1000000 AS BIGINT))
            WHEN regexp_full_match(client_time, '\\d{{4}}-\\d{{2}}-\\d{{2}}.*')
                THEN try_cast(replace(replace(client_time, 'T', ' '), 'Z', '') AS TIMESTAMP)
        END AS ts,
        try_cast(dur AS DOUBLE) AS dur
    FROM raw
)
SELECT *,
       ts IS NOT NULL AND strftime(ts, '%Y-%m') = month AS in_month
FROM parsed
""")
con.sql("SELECT * FROM ev LIMIT 5").df()

## 1. Coverage: số event / session / device / customer theo tháng × platform

In [ ]:
coverage = con.sql("""
    SELECT month, platform,
           count(*)                     AS events,
           count(DISTINCT session_id)   AS sessions,
           count(DISTINCT device_id)    AS devices,
           count(DISTINCT customer_id)  AS customers,
           avg((customer_id IS NULL)::INT) * 100 AS pct_events_no_customer,
           avg((ts IS NULL)::INT) * 100          AS pct_bad_ts,
           avg((NOT in_month)::INT) * 100        AS pct_out_of_month
    FROM ev
    GROUP BY ALL
    ORDER BY month, platform
""").df()

# make sure missing (month, platform) combos are visible as 0 rows
grid = pd.MultiIndex.from_product([MONTHS, PLATFORMS], names=["month", "platform"])
coverage = coverage.set_index(["month", "platform"]).reindex(grid).reset_index()
coverage["events_per_session"] = coverage["events"] / coverage["sessions"]
coverage["sessions_per_device"] = coverage["sessions"] / coverage["devices"]
coverage

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, col in zip(axes, ["events", "sessions", "devices", "customers"]):
    sns.barplot(data=coverage.fillna({col: 0}), x="month", y=col, hue="platform",
                palette=PALETTE, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}K"))
fig.suptitle("Volume by month × platform (missing bar = platform không có dữ liệu tháng đó)")
plt.tight_layout()

## 2. Chất lượng timestamp

Phân bố `ts` theo năm-tháng thực tế cho từng file tháng. Phần lớn event nằm đúng tháng; phần còn lại là
clock lệch của thiết bị (năm 1970, 2036, ...) hoặc event gửi trễ.

In [ ]:
ts_quality = con.sql("""
    SELECT month, platform,
           CASE WHEN ts IS NULL THEN 'unparsable'
                WHEN in_month THEN 'in month'
                WHEN strftime(ts, '%Y-%m') < month THEN 'earlier'
                ELSE 'later' END AS bucket,
           count(*) AS events
    FROM ev GROUP BY ALL
""").df()
ts_quality["pct"] = ts_quality["events"] / ts_quality.groupby(["month", "platform"])["events"].transform("sum") * 100
ts_quality.pivot_table(index=["month", "platform"], columns="bucket", values="pct", fill_value=0)

In [ ]:
bad_examples = con.sql("""
    SELECT month, platform, client_time, count(*) AS n
    FROM ev WHERE ts IS NULL
    GROUP BY ALL ORDER BY n DESC LIMIT 15
""").df()
bad_examples

## 3. Phân phối `key` (view / action)

Từ đây trở đi các phân tích theo thời gian chỉ dùng event `in_month = true`.

In [ ]:
key_dist = con.sql("""
    SELECT month, platform, key, count(*) AS events
    FROM ev GROUP BY ALL
""").df()
key_dist["pct"] = key_dist["events"] / key_dist.groupby(["month", "platform"])["events"].transform("sum") * 100
display(key_dist.pivot_table(index=["month", "platform"], columns="key", values="pct", fill_value=0))

g = sns.catplot(data=key_dist, x="month", y="pct", hue="key", col="platform", kind="bar", order=MONTHS,
                height=4, aspect=1.4, col_order=PLATFORMS)
g.set_axis_labels("", "% events")
g.fig.suptitle("view / action share", y=1.03);

## 4. Phân phối theo thời gian: theo ngày và theo giờ trong ngày (giờ VN)

In [ ]:
daily = con.sql("""
    SELECT month, platform, CAST(ts AS DATE) AS day,
           count(*) AS events, count(DISTINCT session_id) AS sessions
    FROM ev WHERE in_month
    GROUP BY ALL ORDER BY day
""").df()
daily["day_of_month"] = pd.to_datetime(daily["day"]).dt.day

fig, axes = plt.subplots(1, 2, figsize=(18, 4.5), sharex=True)
for ax, platform in zip(axes, PLATFORMS):
    sub = daily[daily["platform"] == platform]
    sns.lineplot(data=sub, x="day_of_month", y="sessions", hue="month", hue_order=MONTHS, marker="o", ax=ax)
    ax.set_title(f"{platform}: sessions per day")
plt.tight_layout()

In [ ]:
hourly = con.sql(f"""
    SELECT month, platform,
           CAST(ts + INTERVAL {LOCAL_TZ_OFFSET_HOURS} HOUR AS DATE) AS local_day,
           hour(ts + INTERVAL {LOCAL_TZ_OFFSET_HOURS} HOUR) AS hour,
           isodow(ts + INTERVAL {LOCAL_TZ_OFFSET_HOURS} HOUR) AS dow,
           count(*) AS events
    FROM ev WHERE in_month
    GROUP BY ALL
""").df()

by_hour = hourly.groupby(["month", "platform", "hour"], as_index=False)["events"].sum()
by_hour["pct"] = by_hour["events"] / by_hour.groupby(["month", "platform"])["events"].transform("sum") * 100
# a month has 4 or 5 of each weekday, so average per calendar day before comparing weekdays
by_dow = (hourly.groupby(["month", "platform", "dow", "local_day"], as_index=False)["events"].sum()
          .groupby(["month", "platform", "dow"], as_index=False)["events"].mean())
by_dow["pct"] = by_dow["events"] / by_dow.groupby(["month", "platform"])["events"].transform("sum") * 100

fig, axes = plt.subplots(2, 2, figsize=(18, 8))
for j, platform in enumerate(PLATFORMS):
    sns.lineplot(data=by_hour[by_hour["platform"] == platform], x="hour", y="pct", hue="month",
                 hue_order=MONTHS, marker="o", ax=axes[0, j])
    axes[0, j].set_title(f"{platform}: % events by hour (UTC+{LOCAL_TZ_OFFSET_HOURS})")
    sns.barplot(data=by_dow[by_dow["platform"] == platform], x="dow", y="pct", hue="month",
                hue_order=MONTHS, ax=axes[1, j])
    axes[1, j].set_title(f"{platform}: avg events per weekday, % of month (1=Mon)")
plt.tight_layout()

## 5. Phân phối cấp session

Materialize bảng `sessions` một lần (group theo `month, platform, session_id`) để các phân tích sau chạy nhanh.
Build từng (month, platform) một để tránh out-of-memory (máy 16GB).
`span_min` = thời gian từ event đầu đến event cuối (chỉ tính event có `ts` hợp lệ trong tháng).

In [ ]:
SESSION_SQL = """
SELECT month, platform, session_id,
       any_value(device_id)                  AS device_id,
       count(*)                              AS n_events,
       count(*) FILTER (WHERE key = 'view')   AS n_view,
       count(*) FILTER (WHERE key = 'action') AS n_action,
       count(DISTINCT segmentation_name)     AS n_unique_screens,
       date_diff('second', min(ts) FILTER (WHERE in_month), max(ts) FILTER (WHERE in_month)) / 60.0 AS span_min
FROM ev
WHERE month = $month AND platform = $platform
GROUP BY ALL
"""
available = coverage.dropna(subset=["events"])[["month", "platform"]].itertuples(index=False)
for i, (month, platform) in enumerate(available):
    params = {"month": month, "platform": platform}
    if i == 0:
        con.execute(f"CREATE OR REPLACE TABLE sessions AS {SESSION_SQL}", params)
    else:
        con.execute(f"INSERT INTO sessions {SESSION_SQL}", params)
    print(f"sessions built: {month} {platform}")
con.sql("SELECT month, platform, count(*) AS sessions FROM sessions GROUP BY ALL ORDER BY ALL").df()

In [ ]:
QUANTILES = [0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

def quantile_table(metric: str, table: str = "sessions", where: str = "TRUE") -> pd.DataFrame:
    q_list = ", ".join(str(q) for q in QUANTILES)
    df = con.sql(f"""
        SELECT month, platform, count(*) AS n, avg({metric}) AS mean,
               quantile_cont({metric}, [{q_list}]) AS q, max({metric}) AS max
        FROM {table} WHERE {metric} IS NOT NULL AND {where}
        GROUP BY ALL ORDER BY month, platform
    """).df()
    q = pd.DataFrame(df.pop("q").tolist(), columns=[f"p{int(x * 100)}" for x in QUANTILES])
    return pd.concat([df, q], axis=1)

for metric in ["n_events", "n_view", "n_action", "n_unique_screens", "span_min"]:
    print(f"\n=== {metric} ===")
    display(quantile_table(metric))

In [ ]:
def session_ecdf(metric: str, title: str, round_to: int = 0, xlim: tuple | None = None):
    """ECDF (% sessions <= x) per month × platform, aggregated in DuckDB on rounded values."""
    df = con.sql(f"""
        SELECT month, platform, round({metric}, {round_to}) AS x, count(*) AS n
        FROM sessions WHERE {metric} IS NOT NULL
        GROUP BY ALL
    """).df().sort_values(["month", "platform", "x"])
    df["ecdf"] = df.groupby(["month", "platform"])["n"].cumsum() / df.groupby(["month", "platform"])["n"].transform("sum") * 100
    df = df[df["x"] > 0]
    fig, axes = plt.subplots(1, 2, figsize=(18, 4.5), sharey=True)
    for ax, platform in zip(axes, PLATFORMS):
        sns.lineplot(data=df[df["platform"] == platform], x="x", y="ecdf", hue="month", hue_order=MONTHS,
                     drawstyle="steps-post", ax=ax)
        ax.set_xscale("log")
        if xlim:
            ax.set_xlim(*xlim)
        ax.set_title(f"{platform}: {title}")
        ax.set_xlabel(metric)
        ax.set_ylabel("% sessions <= x")
    plt.tight_layout()

session_ecdf("n_events", "ECDF of events per session", xlim=(1, 5_000))
session_ecdf("n_unique_screens", "ECDF of unique segmentation_name per session", xlim=(1, 300))
session_ecdf("span_min", "ECDF of session span (minutes)", round_to=2, xlim=(0.01, 50_000))

In [ ]:
# share of very short sessions, useful for later filtering decisions
con.sql("""
    SELECT month, platform,
           avg((n_events = 1)::INT) * 100        AS pct_1_event,
           avg((n_events <= 3)::INT) * 100       AS pct_le_3_events,
           avg((n_action = 0)::INT) * 100        AS pct_no_action,
           avg((span_min < 0.5)::INT) * 100      AS pct_span_lt_30s,
           avg((span_min > 60)::INT) * 100       AS pct_span_gt_1h
    FROM sessions GROUP BY ALL ORDER BY ALL
""").df()

## 6. Phân phối cấp device: số session / device

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE devices AS
SELECT month, platform, device_id, count(*) AS n_sessions, sum(n_events) AS n_events
FROM sessions GROUP BY ALL
""")
display(quantile_table("n_sessions", table="devices"))
display(quantile_table("n_events", table="devices"))

## 7. `dur` (chỉ có ở T6-T8)

`dur` không có trong file T3 nên tháng này sẽ không xuất hiện, và chỉ có giá trị ở event `view`.
Cột này rất bẩn:
- có giá trị cỡ `1e11`-`1.7e12` (timestamp bị ghi vào `dur`);
- đơn vị có vẻ khác nhau giữa platform (android median ~1-2, ios median ~450) nên **không so sánh trực tiếp
  android với ios** trước khi xác nhận đơn vị.

Bảng 1 là raw, bảng 2 là tỉ lệ giá trị bất thường, bảng 3 bỏ các giá trị âm / giống timestamp (`> DUR_TIMESTAMP_LIKE`).

In [ ]:
DUR_TIMESTAMP_LIKE = 1e9

display(quantile_table("dur", table="ev"))
display(con.sql(f"""
    SELECT month, platform, key,
           avg((dur IS NULL)::INT) * 100                   AS pct_null,
           avg((dur < 0)::INT) * 100                       AS pct_negative,
           avg((dur > {DUR_TIMESTAMP_LIKE})::INT) * 100    AS pct_timestamp_like
    FROM ev WHERE month >= '2026-06'
    GROUP BY ALL ORDER BY ALL
""").df())
display(quantile_table("dur", table="ev", where=f"dur BETWEEN 0 AND {DUR_TIMESTAMP_LIKE}"))

## 8. Top screens / segmentation_name theo platform và tháng

In [ ]:
screen_counts = pd.concat([
    con.execute("""
        SELECT month, platform, key, segmentation_name, count(*) AS events,
               count(DISTINCT session_id) AS sessions
        FROM ev WHERE month = $month AND platform = $platform
        GROUP BY ALL
    """, {"month": m, "platform": p}).df()
    for m, p in coverage.dropna(subset=["events"])[["month", "platform"]].itertuples(index=False)
], ignore_index=True)
screen_counts["pct_events"] = (
    screen_counts["events"] / screen_counts.groupby(["month", "platform"])["events"].transform("sum") * 100
)
session_totals = coverage.set_index(["month", "platform"])["sessions"]
screen_counts["pct_sessions"] = (
    screen_counts["sessions"] / session_totals.reindex(
        pd.MultiIndex.from_frame(screen_counts[["month", "platform"]])).to_numpy() * 100
)

# number of distinct names per month × platform × key
screen_counts.groupby(["month", "platform", "key"])["segmentation_name"].nunique().unstack("key")

In [ ]:
def top_screen_heatmap(platform: str, key: str, value: str = "pct_sessions", n: int = TOP_N_SCREENS):
    sub = screen_counts[(screen_counts["platform"] == platform) & (screen_counts["key"] == key)]
    if sub.empty:
        return
    top = sub.groupby("segmentation_name")[value].mean().nlargest(n).index
    mat = sub[sub["segmentation_name"].isin(top)].pivot_table(
        index="segmentation_name", columns="month", values=value).reindex(index=top, columns=MONTHS)
    fig, ax = plt.subplots(figsize=(16, 0.35 * n + 1.5))
    sns.heatmap(mat, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": value}, ax=ax)
    ax.set_title(f"{platform} / {key}: top {n} by mean {value} (blank = absent that month)")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()

for platform in PLATFORMS:
    for key in ["view", "action"]:
        top_screen_heatmap(platform, key)

## 9. Độ ổn định vocabulary giữa các tháng

Với mỗi platform: bao nhiêu `segmentation_name` của tháng này đã xuất hiện ở tháng trước,
và tỉ lệ event thuộc về các name mới. Hữu ích để biết model train trên một tháng có áp dụng được cho tháng khác không.

In [ ]:
rows = []
for platform in PLATFORMS:
    months = [m for m in MONTHS
              if not screen_counts[(screen_counts["platform"] == platform) & (screen_counts["month"] == m)].empty]
    for prev, cur in zip(months, months[1:]):
        prev_df = screen_counts[(screen_counts["platform"] == platform) & (screen_counts["month"] == prev)]
        cur_df = screen_counts[(screen_counts["platform"] == platform) & (screen_counts["month"] == cur)]
        prev_names = set(zip(prev_df["key"], prev_df["segmentation_name"]))
        cur_names = set(zip(cur_df["key"], cur_df["segmentation_name"]))
        is_new = [(k, s) not in prev_names for k, s in zip(cur_df["key"], cur_df["segmentation_name"])]
        rows.append({
            "platform": platform, "from": prev, "to": cur,
            "names_prev": len(prev_names), "names_cur": len(cur_names),
            "new_names": len(cur_names - prev_names), "dropped_names": len(prev_names - cur_names),
            "pct_events_on_new_names": cur_df.loc[is_new, "events"].sum() / cur_df["events"].sum() * 100,
        })
pd.DataFrame(rows)

In [ ]:
# biggest share shifts between consecutive months (per platform, per key)
shift_rows = []
for platform in PLATFORMS:
    wide = screen_counts[screen_counts["platform"] == platform].pivot_table(
        index=["key", "segmentation_name"], columns="month", values="pct_events", fill_value=0)
    cols = [m for m in MONTHS if m in wide.columns]
    for prev, cur in zip(cols, cols[1:]):
        delta = (wide[cur] - wide[prev]).rename("delta_pct_points")
        top = delta.abs().nlargest(10).index
        out = pd.DataFrame({prev: wide.loc[top, prev], cur: wide.loc[top, cur], "delta": delta.loc[top]})
        out.insert(0, "transition", f"{platform}: {prev} -> {cur}")
        shift_rows.append(out.reset_index())
pd.concat(shift_rows, ignore_index=True) if shift_rows else "no consecutive months"